In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine, Integer, String, Date, Boolean, Numeric, Table, Column, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from typing import List, Optional
from datetime import date

In [3]:
engine = create_engine("sqlite:///restaurant_backoffice.db")

class Base(DeclarativeBase):
    ...


In [4]:
class Employee(Base):
    __tablename__ = "employee"
    
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(64), nullable=False)
    surname: Mapped[str] = mapped_column(String(64), nullable=False)
    national_id: Mapped[str] = mapped_column(String(20), nullable=False, unique=True)
    role: Mapped[str] = mapped_column(String(20), nullable=False)

    # Relació 1:1
    credentials: Mapped[Optional["AppCredential"]] = relationship(back_populates="employee", uselist=False)


    # Relació cap a Sale (1:N)
    sales: Mapped[list["Sale"]] = relationship(back_populates="employee")

class AppCredential(Base):
    __tablename__ = "app_credential"
    
    # Foreign Key cap a Employee
    id_employee: Mapped[int] = mapped_column(Integer, ForeignKey("employee.id", ondelete="CASCADE"), primary_key=True)
    
    email: Mapped[str] = mapped_column(String(128), unique=True, nullable=False)
    password_hash: Mapped[str] = mapped_column(String(255), nullable=False)
    
    # Relació inversa cap a employee
    employee: Mapped["Employee"] = relationship(back_populates="credentials")

class Category(Base):
    __tablename__ = "category"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    category_name: Mapped[str] = mapped_column(String(64), nullable=False)

    # Relació: Una categoria té una LLISTA de plats, per això fem servir List["Dish"], significant que és una relació 1:N, en aquest cas estem al costat "One" de la relació. (1 categoria té molts plats)
    dishes: Mapped[list["Dish"]] = relationship(back_populates="category")

# Definició de la taula intermèdia dish_allergen
dish_allergen = Table(
    "dish_allergen",
    Base.metadata,
    Column("id_dish", ForeignKey("dish.id", ondelete="CASCADE"), primary_key=True),
    Column("id_allergen", ForeignKey("allergen.id", ondelete="CASCADE"), primary_key=True),
)

# Definició de la taula intermèdia menu_dish
menu_dish = Table(
    "menu_dish",
    Base.metadata,
    Column("id_menu", ForeignKey("menu.id", ondelete="CASCADE"), primary_key=True),
    Column("id_dish", ForeignKey("dish.id", ondelete="CASCADE"), primary_key=True),
)

class Dish(Base):
    __tablename__ = "dish"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    dish_name: Mapped[str] = mapped_column(String(64), nullable=False)
    description: Mapped[Optional[str]] = mapped_column(String)
    is_active: Mapped[bool] = mapped_column(default=True)
    
    # La Foreign Key sempre va al costat "Many"
    id_category: Mapped[int] = mapped_column(ForeignKey("category.id"), nullable=False)
    
    # Relació N:1 (Many-to-One): Un plat pertany a UNA sola categoria,
    # però una categoria pot tenir molts plats.
    category: Mapped["Category"] = relationship(back_populates="dishes")

    # Relacions Many-to-Many amb Allergen i Menu
    allergens: Mapped[list["Allergen"]] = relationship(
        secondary=dish_allergen, back_populates="dishes"
    )
    menus: Mapped[list["Menu"]] = relationship(
        secondary=menu_dish, back_populates="dishes"
    )

class Allergen(Base):
    __tablename__ = "allergen"

    id: Mapped[int] = mapped_column(primary_key=True)
    allergen_name: Mapped[str] = mapped_column(String(64), nullable=False)

    # Relació Many-to-Many amb Dish
    # 'secondary' indica la taula associativa que connecta Dish i Allergen
    # 'back_populates' indica el nom de l'atribut a l'altra classe que fa referència a aquesta relació
    dishes: Mapped[list["Dish"]] = relationship(
        secondary=dish_allergen, back_populates="allergens"
    )

class Menu(Base):
    __tablename__ = "menu"

    id: Mapped[int] = mapped_column(primary_key=True)
    menu_name: Mapped[str] = mapped_column(String(64), nullable=False)
    price: Mapped[float] = mapped_column(Numeric(4, 2), nullable=False)

    # Relació Many-to-Many amb Dish
    dishes: Mapped[list["Dish"]] = relationship(
        secondary=menu_dish, back_populates="menus"
    )

    # Relació 1:N amb Sale (un menú pot estar en moltes vendes, però una venda només té un menú)
    sales: Mapped[list["Sale"]] = relationship(back_populates="menu")

class Sale(Base):
    __tablename__ = "sale"

    # Clau primària composta per id_menu i id_employee
    id_menu: Mapped[int] = mapped_column(ForeignKey("menu.id"), primary_key=True)
    id_employee: Mapped[int] = mapped_column(ForeignKey("employee.id"), primary_key=True)

    # Atributs adicionals de la relació N:M 
    sale_date: Mapped[date] = mapped_column(Date, primary_key=True) # Afegit a PK si un mateix empleat ven el mateix menú en dies diferents
    quantity_sold: Mapped[int] = mapped_column(Integer, default=1) # Afegin un default per facilitar la creació de vendes quan només es ven una unitat

    # Relació cap a Menu i Employee
    menu: Mapped["Menu"] = relationship(back_populates="sales")
    employee: Mapped["Employee"] = relationship(back_populates="sales")

# 4. Creació de les taules a la base de dades
Base.metadata.create_all(engine)

In [5]:
#Printar base de dades
print(Base.metadata.tables.keys())

dict_keys(['employee', 'app_credential', 'category', 'dish_allergen', 'menu_dish', 'dish', 'allergen', 'menu', 'sale'])


In [1]:
#Proves sobre la base de dades
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")

In [2]:
from domain import (
    Base,
    engine,
    Session,
    Employee,
    AppCredential,
    Category,
    Allergen,
    Dish,
    Menu,
    Sale
)

In [3]:
#Cel·la 1: Configuració i Càrrega d'Elements Bàsics (Categories i Al·lèrgens)
#Copia i enganxa això a la següent cel·la del teu Notebook. Això omplirà les dades "base" necessàries per crear plats.

with Session() as session:
    # ----------------------------
    # 1. CATEGORIES
    # ----------------------------
    category_names = ["Entrants", "Plats Principals", "Postres", "Begudes"]

    existing_categories = {c.category_name: c for c in session.query(Category).all()}
    inserted_cat = 0
    for name in category_names:
        if name not in existing_categories:
            category = Category(category_name=name)
            session.add(category)
            session.flush() # Ens assegurem de tenir l'ID
            existing_categories[name] = category
            inserted_cat += 1

    # ----------------------------
    # 2. ALLERGENS
    # ----------------------------
    allergen_names = ["Gluten", "Lactis", "Fruits secs", "Crustacis", "Ou", "Soja"]

    existing_allergens = {a.allergen_name: a for a in session.query(Allergen).all()}
    inserted_all = 0
    for name in allergen_names:
        if name not in existing_allergens:
            allergen = Allergen(allergen_name=name)
            session.add(allergen)
            session.flush()
            existing_allergens[name] = allergen
            inserted_all += 1

    session.commit()
    print(f"Inserides {inserted_cat} categories i {inserted_all} al·lèrgens.")

Inserides 4 categories i 6 al·lèrgens.


In [4]:
#Cel·la 2: Empleats i Credencials
#Aquest bloc crea els empleats i aprofita la relació 1:1 per generar les credencials de l'aplicació.

with Session() as session:
    # ----------------------------
    # 3. EMPLOYEES & CREDENTIALS
    # ----------------------------
    employees_data = [
        ("Joan", "Pérez", "11111111A", "Cambrer", "joan@restaurant.cat", "hash_joan_123"),
        ("Maria", "García", "22222222B", "Cuinera", "maria@restaurant.cat", "hash_maria_456"),
        ("Marc", "Vila", "33333333C", "Encarregat", "marc@restaurant.cat", "hash_marc_789"),
    ]

    existing_employees = {e.national_id for (e,) in session.query(Employee.national_id).all()}
    inserted_emp = 0

    for name, surname, national_id, role, email, password_hash in employees_data:
        if national_id not in existing_employees:
            employee = Employee(
                name=name,
                surname=surname,
                national_id=national_id,
                role=role
            )

            # Creació lligada a la relació 1:1
            employee.credentials = AppCredential(
                email=email,
                password_hash=password_hash
            )
            session.add(employee)
            inserted_emp += 1

    session.commit()
    print(f"Inserits {inserted_emp} empleats amb les seves credencials.")

Inserits 3 empleats amb les seves credencials.


In [5]:
#Cel·la 3: Plats (Dishes)
#Aquí ja relacionem les categories creades a la Cel·la 1 amb la taula N:M dels al·lèrgens.

with Session() as session:
    # ----------------------------
    # 4. DISHES
    # ----------------------------
    categories = {c.category_name: c for c in session.query(Category).all()}
    allergens = {a.allergen_name: a for a in session.query(Allergen).all()}

    dishes_data = [
        {
            "name": "Amanida Cèsar",
            "desc": "Amanida fresca amb pollastre i crostons",
            "category": "Entrants",
            "allergens": ["Gluten", "Lactis", "Ou"]
        },
        {
            "name": "Macarrons a la bolonyesa",
            "desc": "Pasta amb salsa de tomàquet i carn picada",
            "category": "Plats Principals",
            "allergens": ["Gluten"]
        },
        {
            "name": "Lluç al forn",
            "desc": "Lluç fresc amb patates fornades",
            "category": "Plats Principals",
            "allergens": []
        },
        {
            "name": "Pastís de formatge",
            "desc": "Cheesecake casolà",
            "category": "Postres",
            "allergens": ["Lactis", "Ou", "Gluten"]
        },
        {
            "name": "Aigua mineral",
            "desc": "Ampolla de 500ml",
            "category": "Begudes",
            "allergens": []
        }
    ]

    existing_dishes = {d.dish_name: d for d in session.query(Dish).all()}
    inserted_dishes = 0

    for item in dishes_data:
        if item["name"] not in existing_dishes:
            dish = Dish(
                dish_name=item["name"],
                description=item["desc"],
                is_active=True,
                id_category=categories[item["category"]].id
            )
            # Associem al·lèrgens (Relació Many-to-Many)
            dish.allergens = [allergens[a] for a in item["allergens"]]

            session.add(dish)
            inserted_dishes += 1

    session.commit()
    print(f"Inserits {inserted_dishes} plats amb els seus al·lèrgens.")

Inserits 5 plats amb els seus al·lèrgens.


In [6]:
#Cel·la 4: Menús
#Agrupem els plats existents per formar diferents menús.
with Session() as session:
    # ----------------------------
    # 5. MENUS
    # ----------------------------
    dishes = {d.dish_name: d for d in session.query(Dish).all()}

    menus_data = [
        ("Menú Diari", 12.50, ["Amanida Cèsar", "Macarrons a la bolonyesa", "Aigua mineral"]),
        ("Menú Nit", 18.00, ["Amanida Cèsar", "Lluç al forn", "Pastís de formatge", "Aigua mineral"]),
    ]

    existing_menus = {m.menu_name: m for m in session.query(Menu).all()}
    inserted_menus = 0

    for menu_name, price, menu_dishes in menus_data:
        if menu_name not in existing_menus:
            menu = Menu(
                menu_name=menu_name,
                price=price
            )
            # Associem plats al menú (Relació Many-to-Many)
            menu.dishes = [dishes[d_name] for d_name in menu_dishes]

            session.add(menu)
            inserted_menus += 1

    session.commit()
    print(f"Inserits {inserted_menus} menús.")

Inserits 2 menús.


In [7]:
#Cel·la 5: Vendes (Sales)
#L'últim bloc simula vendes reals per part dels empleats, afegint la data i la quantitat venuda.

from datetime import date

with Session() as session:
    # ----------------------------
    # 6. SALES
    # ----------------------------
    menus = {m.menu_name: m for m in session.query(Menu).all()}
    employees = {e.national_id: e for e in session.query(Employee).all()}

    # Data: (Nom Menú, DNI Empleat, Data, Quantitat)
    sales_data = [
        ("Menú Diari", "11111111A", date(2026, 3, 26), 2),
        ("Menú Nit", "11111111A", date(2026, 3, 26), 1),
        ("Menú Diari", "33333333C", date(2026, 3, 25), 4),
    ]

    existing_sales = {
        (s.id_menu, s.id_employee, s.sale_date)
        for s in session.query(Sale).all()
    }
    inserted_sales = 0

    for menu_name, emp_dni, sale_date, quantity in sales_data:
        menu = menus.get(menu_name)
        employee = employees.get(emp_dni)

        if not menu or not employee:
            print(f"Avís: Menú '{menu_name}' o Empleat '{emp_dni}' no trobat.")
            continue

        key = (menu.id, employee.id, sale_date)
        if key in existing_sales:
            continue

        sale = Sale(
            id_menu=menu.id,
            id_employee=employee.id,
            sale_date=sale_date,
            quantity_sold=quantity
        )
        session.add(sale)
        inserted_sales += 1

    session.commit()
    print(f"Inserides {inserted_sales} vendes de menús.")

Inserides 3 vendes de menús.
